In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd

df = pd.read_excel("/civilens_dataset_15000_balanced (1).xlsx")

df.head()

,text,category
0,Fake job offer scam,Police
1,Road rage incident,Police
2,Illegal gambling in playground,Police
3,Vehicle vandalized,Police
4,Drunk person creating trouble,Police


In [10]:
df['category'].value_counts()

,count
category,
Police,5000
Municipal,5000
Electricity,5000


In [11]:
 X= df[['text']]
 y = df[['category']]
 y[0:9]

,category
0,Police
1,Police
2,Police
3,Police
4,Police
5,Police
6,Police
7,Police
8,Police


In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2")

X_embeddings = embedding_model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [13]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


# Encode labels
le = LabelEncoder()
y = le.fit_transform(df["category"])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)

In [14]:
from sklearn.svm import LinearSVC
model = LinearSVC()
model.fit(X_train, y_train)


LinearSVC()

In [15]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test,y_pred)
print(accuracy_score(y_train, model.predict(X_train)))
print(accuracy_score(y_test,y_pred))
cm

1.0
1.0


array([[ 993,    0,    0],
       [   0,  970,    0],
       [   0,    0, 1037]])

In [16]:

text = input("Enter complaint: ").strip().lower()

# -------- INVALID PATTERNS --------
invalid_phrases = [
    "my heart is missing",
    "i heart is missing"
]

body_parts = [
    "eye","eyes","hand","hands","leg","legs","arm","arms",
    "head","ear","ears","nose","mouth","finger","fingers",
    "foot","feet","brain","stomach"
]

# -------- VALIDATION FIRST --------
if text in invalid_phrases:
    print("Invalid complaint. Please describe your issue clearly.")

elif "my" in text and "missing" in text and any(bp in text for bp in body_parts):
    print("Invalid complaint. Please describe your issue clearly.")

elif len(text.split()) < 3:
    print("Invalid complaint. Please describe your issue clearly.")

else:
    # -------- EMBEDDING --------
    emb = embedding_model.encode([text])
    pred = model.predict(emb)
    dept = le.inverse_transform(pred)[0]
    print(dept, "dept")

Enter complaint: my wallet is missing
Police dept
